# McDonald's diet problem: read the data from CSV

This notebook solves the same minimum-cost diet LP as `03-McDonaldsDiet.ipynb`, but reads the foods, costs, nutrient amounts, and minimum requirements from a CSV file. The model adapts to the number of foods and nutrients in the selected file.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and run the cells from top to bottom. CSV, DataFrames, NamedArrays, JuMP, HiGHS, and Printf are already included in the course environment.

## Choose a dataset

Edit **only the filename** assigned to `dataset_filename` in the next cell:

- `"mcdonalds.csv"` uses the original classroom data: 9 foods and 7 nutrients.
- `"diet-synthetic.csv"` uses reproducible, generated data: 100 fictional foods and 20 fictional nutrients, in arbitrary units. This dataset has a feasible solution.

After changing the filename, choose **Run All** to reload the data and rebuild the model. A run solves only the selected dataset. To keep your edits, save a personal copy in `student-work/` and leave the published course notebook as the template.

Both files are already in the repository's `data/` folder. The path below uses the active course project, so it works from the repository root, `notebooks/`, or a personal copy in `student-work/`. Keep the course project active. If a file is missing, run **ISyE 524: Update course repository** to download the published notebook and data together.

In [ ]:
using CSV, DataFrames, NamedArrays, JuMP, HiGHS, Printf
import MathOptInterface as MOI

dataset_filename = "mcdonalds.csv"

project_file = Base.active_project()
isnothing(project_file) && error("Activate the course Julia environment first.")
data_path = joinpath(dirname(project_file), "data", dataset_filename)
isfile(data_path) || error(
    "Cannot find $(data_path). Check the filename, activate the course project, and update the course repository."
)

df = CSV.read(data_path, DataFrame)
println("Selected dataset: ", dataset_filename)
println("Foods: ", ncol(df) - 2, "; nutrients: ", nrow(df) - 1)
df

## Understand the CSV layout

A `DataFrame` is a table with named columns. Both CSV files have the same layout:

| Row | `Nutrient` column | `Required` column | Food columns |
| --- | --- | --- | --- |
| First data row | `Cost` | Empty | Cost per serving |
| Remaining rows | Nutrient name | Minimum required | Nutrient amount per serving |

For example, the McDonald's protein row requires at least 55 units, and its Quarter Pounder column supplies 28 units per serving. Each nutrient minimum uses the same units as the coefficients in that row.

`propertynames(df)[3:end]` gets the food-column names as Julia symbols. A name containing spaces, such as `Quarter Pounder`, still works as an index. `df[2:end, :Nutrient]` gets the nutrient labels, and `Matrix{Float64}` extracts a numerical matrix. The blank cell in the `Cost` row is intentional; it is excluded when we read the nutrient minimums.

In [ ]:
nrow(df) >= 2 && ncol(df) >= 3 || error("The CSV needs a cost row, nutrient rows, and food columns.")
names(df)[1:2] == ["Nutrient", "Required"] || error("The first two columns must be Nutrient and Required.")
df[1, :Nutrient] == "Cost" || error("The first data row must contain Cost.")

foods = propertynames(df)[3:end]
nutrients = String.(df[2:end, :Nutrient])

cost = Dict(zip(foods, Float64.(collect(df[1, 3:end]))))
required = Dict(zip(nutrients, Float64.(df[2:end, :Required])))
A = Matrix{Float64}(df[2:end, 3:end])
A_NA = NamedArray(A, (nutrients, foods), ("Nutrients", "Foods"))
A_NA

## Build the linear program

Let $F$ and $N$ be the foods and nutrients read from the CSV. For food $j$, let $x_j$ be its number of servings and $c_j$ its cost. Let $a_{ij}$ be its amount of nutrient $i$, and let $b_i$ be the minimum requirement.

$$
\begin{aligned}
\min_x\quad & \sum_{j\in F} c_j x_j \\
\text{subject to}\quad & \sum_{j\in F} a_{ij}x_j \geq b_i && \forall i\in N, \\
& x_j \geq 0 && \forall j\in F.
\end{aligned}
$$

The model permits fractional servings and imposes lower bounds on all nutrients. It has no upper limits on nutrient intake. Every index set and coefficient comes from the file, so the same code handles either dataset.

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[foods] >= 0)
@objective(model, Min, sum(cost[j] * x[j] for j in foods))
@constraint(model, nutrient_minimum[i in nutrients],
    sum(A_NA[i, j] * x[j] for j in foods) >= required[i])

println("Built a model with ", length(foods), " food variables and ",
    length(nutrients), " nutrient constraints.")

## Solve, check, and report

Check the solver status before reading objective or variable values. The conditional dictionary comprehension keeps foods with more than `1e-6` servings, to omit numerical values close to zero. Iterate over the original food order to make the report easy to compare with the CSV.

In [ ]:
optimize!(model)
status = termination_status(model)
status == MOI.OPTIMAL || error("HiGHS stopped with status $(status).")
is_solved_and_feasible(model) || error("No feasible optimal solution is available.")

minimum_cost = objective_value(model)
solution = Dict(j => value(x[j]) for j in foods if value(x[j]) > 1e-6)

println("Termination status: ", status)
@printf("\nMinimum cost menu for %s is \$%.2f\n", dataset_filename, minimum_cost)
for j in foods
    if haskey(solution, j)
        @printf("Eat %.2f servings of %s\n", solution[j], j)
    end
end

## Check nutrient totals

With `mcdonalds.csv`, the minimum cost is approximately **14.86 dollars**, matching the earlier notebook. The synthetic dataset has different foods, nutrient requirements, and an optimum of its own.

The displayed servings are rounded, but nutrient totals below use the unrounded solution. Rounding servings can violate a nutrient minimum. Which constraints are met exactly, and which have a surplus?

In [ ]:
nutrient_totals = [
    sum(A_NA[i, j] * value(x[j]) for j in foods) for i in nutrients
]
minimums = [required[i] for i in nutrients]
nutrient_report = DataFrame(
    nutrient = nutrients,
    total = nutrient_totals,
    minimum = minimums,
    surplus = nutrient_totals - minimums,
)
nutrient_report